# Logistic Regression — Healthcare Classification (Breast Cancer)

Binary classification with proper preprocessing, CV, tuning, and calibrated evaluation.
Dataset: `sklearn.datasets.load_breast_cancer` (real-world).

**Author:** Olivier Robert-Duboille

**What you'll practice**
- reproducible data loading
- EDA with clear plots
- feature engineering / preprocessing pipelines
- cross-validation + hyperparameter tuning
- baseline vs advanced model comparison

This notebook is part of the *advanced-ml-mastery-collection* and is designed to be reproducible and portfolio-ready.

In [ ]:
# Reproducibility
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Plot settings
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_context('talk')


## Load dataset

Load data into a pandas DataFrame/Series and perform a first sanity check.

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df['target'] = data.target
display(df.head())
print(df['target'].value_counts())


## EDA

Explore distributions, relationships, and potential issues (missing values, skew, outliers).

In [ ]:
# Class balance
plt.figure(figsize=(5,3))
sns.countplot(x='target', data=df)
plt.title('Class balance (0=malignant, 1=benign)')
plt.show()

# Feature distributions (subset)
cols = [c for c in df.columns if c != 'target'][:8]
df[cols].hist(figsize=(12,8), bins=30)
plt.suptitle('Feature histograms (subset)')
plt.show()


## Baselines and pipelines

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

num_cols = X.columns.tolist()
prep = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
ct = ColumnTransformer([('num', prep, num_cols)])

models = {
    'dummy': DummyClassifier(strategy='most_frequent'),
    'logreg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=SEED),
    'svc_rbf': SVC(kernel='rbf', probability=True, random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'roc_auc': 'roc_auc', 'f1': 'f1', 'accuracy': 'accuracy'}

import pandas as pd
rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', ct), ('model', model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        'model': name,
        'roc_auc_mean': res['test_roc_auc'].mean(),
        'f1_mean': res['test_f1'].mean(),
        'acc_mean': res['test_accuracy'].mean(),
    })

display(pd.DataFrame(rows).sort_values('roc_auc_mean', ascending=False))


## Hyperparameter tuning (LogReg + SVC)

In [ ]:
from sklearn.model_selection import GridSearchCV
import numpy as np

log_pipe = Pipeline([('prep', ct), ('model', LogisticRegression(max_iter=5000, solver='liblinear', random_state=SEED))])
log_grid = {
    'model__C': np.logspace(-3, 3, 13),
    'model__penalty': ['l1', 'l2'],
}
log_gs = GridSearchCV(log_pipe, log_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
log_gs.fit(X_train, y_train)
print('Best LogReg AUC:', log_gs.best_score_)
print('Best params:', log_gs.best_params_)

svc_pipe = Pipeline([('prep', ct), ('model', SVC(kernel='rbf', probability=True, random_state=SEED))])
svc_grid = {
    'model__C': np.logspace(-2, 2, 9),
    'model__gamma': np.logspace(-4, 0, 9),
}
svc_gs = GridSearchCV(svc_pipe, svc_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
svc_gs.fit(X_train, y_train)
print('Best SVC AUC:', svc_gs.best_score_)
print('Best params:', svc_gs.best_params_)


## Final evaluation + ROC/Confusion

Evaluate using threshold-free and thresholded metrics; visualize ROC + confusion matrix.

In [ ]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay, classification_report

best = svc_gs.best_estimator_
best.fit(X_train, y_train)
proba = best.predict_proba(X_test)[:, 1]
pred = best.predict(X_test)

print(classification_report(y_test, pred, digits=3))

RocCurveDisplay.from_predictions(y_test, proba)
plt.title('ROC curve (test)')
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title('Confusion matrix (test)')
plt.show()
